# From Scratch: Build an AI Inference LLM with OpenAI-Compatible API

**Platform**: Kaggle (2x T4 GPU, 30GB RAM, 60GB Disk)

**Features**: 
- English prompt input
- Weather query & directory listing
- OpenAI-compatible API format
- Entirely from-scratch neural network training (no pre-trained models)

---

## Step 0: Environment Setup

In [ ]:
import subprocess, sys
packages = ['torch', 'fastapi', 'uvicorn', 'pydantic']
for pkg in packages:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_mem/1e9:.1f}GB')

## Step 1: Configuration

In [ ]:
import os, json, random, math, time, uuid
from collections import Counter
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

BASE_DIR = '/kaggle/working/custom_llm'
os.makedirs(BASE_DIR, exist_ok=True)

MODEL_CONFIG = {
    'vocab_size': 4096, 'max_seq_len': 256, 'd_model': 512,
    'n_heads': 8, 'n_layers': 6, 'd_ff': 2048, 'dropout': 0.1, 'pad_token_id': 0,
}
TRAIN_CONFIG = {
    'batch_size': 32, 'learning_rate': 3e-4, 'weight_decay': 0.01,
    'epochs': 20, 'warmup_steps': 500, 'max_grad_norm': 1.0,
    'save_every': 5, 'eval_every': 1, 'fp16': True, 'num_workers': 2,
}
DATA_CONFIG = {
    'num_weather_samples': 3000, 'num_directory_samples': 3000,
    'train_ratio': 0.9, 'seed': 42,
}
SPECIAL_TOKENS = {'pad': '<pad>', 'bos': '<bos>', 'eos': '<eos>', 'unk': '<unk>'}
PATHS = {
    'data_dir': os.path.join(BASE_DIR, 'data'),
    'raw_data': os.path.join(BASE_DIR, 'data', 'raw_dataset.json'),
    'train_data': os.path.join(BASE_DIR, 'data', 'train.json'),
    'val_data': os.path.join(BASE_DIR, 'data', 'val.json'),
    'tokenizer_dir': os.path.join(BASE_DIR, 'tokenizer'),
    'tokenizer_file': os.path.join(BASE_DIR, 'tokenizer', 'tokenizer.json'),
    'model_dir': os.path.join(BASE_DIR, 'checkpoints'),
    'best_model': os.path.join(BASE_DIR, 'checkpoints', 'best_model.pt'),
    'log_dir': os.path.join(BASE_DIR, 'logs'),
}
for p in PATHS.values():
    d = os.path.dirname(p) if '.' in os.path.basename(p) else p
    os.makedirs(d, exist_ok=True)

print(f'Config set! Model: {MODEL_CONFIG["n_layers"]}L x {MODEL_CONFIG["d_model"]}D x {MODEL_CONFIG["n_heads"]}H')

## Step 2: Raw Dataset Generation
Synthesize ~6000 training samples: 3000 weather + 3000 directory.

In [ ]:
CITIES = [
    "Beijing","Shanghai","Guangzhou","Shenzhen","Chengdu","Hangzhou","Wuhan",
    "Xi'an","Nanjing","Chongqing","Tianjin","Suzhou","Qingdao","Dalian","Xiamen",
    "New York","London","Tokyo","Paris","Berlin","Sydney","Toronto","Moscow",
    "Dubai","Singapore","Seoul","Mumbai","Bangkok","Cairo","Rome","Madrid",
    "Amsterdam","Vienna","Prague","Budapest","Lisbon","Stockholm","Oslo",
    "Helsinki","Copenhagen","Dublin","Zurich","Geneva","Brussels","Warsaw",
    "Istanbul","Jakarta","Manila","Kuala Lumpur","Hanoi",
]
WEATHER_TEMPLATES = [
    "What's the weather in {city}?","What is the weather like in {city}?",
    "How is the weather in {city}?","Tell me the weather in {city}.",
    "Show me the weather for {city}.","Get the weather for {city}.",
    "Check weather in {city}.","Weather in {city}.",
    "What's the weather like in {city} today?","What's the current weather in {city}?",
    "Can you tell me the weather in {city}?","Could you check the weather in {city}?",
    "I want to know the weather in {city}.","Please show me the weather in {city}.",
    "Give me the weather update for {city}.","Is it raining in {city}?",
    "Is it sunny in {city}?","How cold is it in {city}?",
    "How hot is it in {city}?","What temperature is it in {city}?",
    "What's the forecast for {city}?","Weather forecast for {city}.",
    "Current conditions in {city}.","What's it like outside in {city}?",
    "Do I need an umbrella in {city}?","Should I wear a jacket in {city}?",
    "Is it going to rain in {city}?","Will it snow in {city}?",
    "How windy is it in {city}?","What's the humidity in {city}?",
]
DIRECTORY_PATHS = [
    "/home/user","/home/user/documents","/home/user/downloads","/home/user/pictures",
    "/home/user/desktop","/home/user/music","/home/user/videos","/var/log",
    "/var/www","/etc","/tmp","/opt","/usr/local","/root","/home/admin",
    "/data","/data/projects","/data/models","/data/logs","/srv/app",
    "/srv/www","/workspace","/workspace/src","C:\\Users\\Admin",
    "C:\\Users\\Admin\\Desktop","C:\\Users\\Admin\\Documents",
    "C:\\Users\\Admin\\Downloads","C:\\Program Files","C:\\Projects",
]
DIRECTORY_TEMPLATES = [
    "List files in {path}.","Show me the files in {path}.",
    "What files are in {path}?","List directory {path}.",
    "Show directory contents of {path}.","List all files in {path}.",
    "What's in {path}?","Display the contents of {path}.",
    "Show what's inside {path}.","List the files under {path}.",
    "Can you list the files in {path}?","Could you show me the contents of {path}?",
    "I want to see the files in {path}.","Please list the directory {path}.",
    "What directories are in {path}?","Show me the folder contents of {path}.",
    "List all items in {path}.","Browse {path}.",
    "Open directory {path}.","Explore {path}.",
    "What is inside the folder {path}?","Show the file listing for {path}.",
    "Display files and folders in {path}.","Get a directory listing for {path}.",
    "What can I find in {path}?","List the contents of the folder {path}.",
    "Show me everything in {path}.","Enumerate files in {path}.",
    "Read the directory {path}.","What does {path} contain?",
]

random.seed(DATA_CONFIG['seed'])
weather_samples = []
for _ in range(DATA_CONFIG['num_weather_samples']):
    city = random.choice(CITIES)
    prompt = random.choice(WEATHER_TEMPLATES).format(city=city)
    output = json.dumps({"name":"get_weather","arguments":{"city":city}}, separators=(",",":"))
    weather_samples.append({"prompt":prompt,"output":output,"task":"weather"})

dir_samples = []
for _ in range(DATA_CONFIG['num_directory_samples']):
    path = random.choice(DIRECTORY_PATHS)
    prompt = random.choice(DIRECTORY_TEMPLATES).format(path=path)
    output = json.dumps({"name":"list_directory","arguments":{"path":path}}, separators=(",",":"))
    dir_samples.append({"prompt":prompt,"output":output,"task":"directory"})

all_samples = weather_samples + dir_samples
random.shuffle(all_samples)
split_idx = int(len(all_samples) * DATA_CONFIG['train_ratio'])
train_samples = all_samples[:split_idx]
val_samples = all_samples[split_idx:]

for k,v in [('raw',all_samples),('train',train_samples),('val',val_samples)]:
    with open(PATHS[f'{k}_data'],'w') as f: json.dump(v,f)

print(f'Dataset: {len(all_samples)} total, {len(train_samples)} train, {len(val_samples)} val')
print(f'Example: {weather_samples[0]["prompt"]} -> {weather_samples[0]["output"]}')

## Step 3: Custom BPE Tokenizer (From Scratch)

In [ ]:
class BPETokenizer:
    def __init__(self, vocab_size=4096):
        self.vocab_size = vocab_size
        self.merges = []
        self.token_to_id = {}
        self.id_to_token = {}
        self.special_tokens = SPECIAL_TOKENS
        self._initialized = False

    @property
    def pad_token_id(self): return self.token_to_id.get(self.special_tokens['pad'], 0)
    @property
    def bos_token_id(self): return self.token_to_id.get(self.special_tokens['bos'], 1)
    @property
    def eos_token_id(self): return self.token_to_id.get(self.special_tokens['eos'], 2)

    def _text_to_bytes(self, text): return list(text.encode('utf-8'))

    def _merge_pair(self, token_list, pair, new_token):
        merged, i = [], 0
        while i < len(token_list):
            if i < len(token_list)-1 and token_list[i]==pair[0] and token_list[i+1]==pair[1]:
                merged.append(new_token); i += 2
            else: merged.append(token_list[i]); i += 1
        return merged

    def train(self, texts, verbose=True):
        special_list = [self.special_tokens['pad'], self.special_tokens['bos'],
                        self.special_tokens['eos'], self.special_tokens['unk']]
        byte_tokens = [f'<0x{b:02X}>' for b in range(256)]
        self.token_to_id, self.id_to_token = {}, {}
        idx = 0
        for st in special_list:
            self.token_to_id[st]=idx; self.id_to_token[idx]=st; idx+=1
        for bt in byte_tokens:
            self.token_to_id[bt]=idx; self.id_to_token[idx]=bt; idx+=1
        tokenized_corpus = []
        for text in texts:
            tokenized_corpus.append([f'<0x{b:02X}>' for b in self._text_to_bytes(text)])
        num_merges = self.vocab_size - idx
        if verbose: print(f'Base vocab: {idx}, merges: {num_merges}')
        for mi in range(num_merges):
            pc = Counter()
            for ts in tokenized_corpus:
                for i in range(len(ts)-1): pc[(ts[i],ts[i+1])] += 1
            if not pc: break
            bp = pc.most_common(1)[0][0]
            nt = bp[0]+bp[1]
            self.merges.append(bp)
            self.token_to_id[nt]=idx; self.id_to_token[idx]=nt; idx+=1
            for j in range(len(tokenized_corpus)):
                tokenized_corpus[j] = self._merge_pair(tokenized_corpus[j], bp, nt)
            if verbose and (mi+1)%200==0: print(f'  Merge {mi+1}/{num_merges}')
        self._initialized = True
        if verbose: print(f'Tokenizer: {len(self.token_to_id)} tokens')

    def encode(self, text):
        assert self._initialized
        if text in self.token_to_id: return [self.token_to_id[text]]
        tokens = [f'<0x{b:02X}>' for b in self._text_to_bytes(text)]
        for pair in self.merges:
            nt = pair[0]+pair[1]
            new_tokens, i = [], 0
            while i < len(tokens):
                if i<len(tokens)-1 and tokens[i]==pair[0] and tokens[i+1]==pair[1]:
                    new_tokens.append(nt); i+=2
                else: new_tokens.append(tokens[i]); i+=1
            tokens = new_tokens
        return [self.token_to_id.get(t, self.token_to_id.get(self.special_tokens['unk'],3)) for t in tokens]

    def decode(self, ids):
        assert self._initialized
        tokens = []
        for id_ in ids:
            if id_ in self.id_to_token:
                t = self.id_to_token[id_]
                if t not in [self.special_tokens['pad'],self.special_tokens['bos'],
                             self.special_tokens['eos'],self.special_tokens['unk']]:
                    tokens.append(t)
        byte_list = []
        for token in tokens:
            if token.startswith('<0x') and token.endswith('>'):
                try: byte_list.append(int(token[3:-1],16))
                except: pass
            else:
                i = 0
                while i < len(token):
                    if token[i:i+4].startswith('<0x') and i+5<len(token) and token[i+4]=='>':
                        try: byte_list.append(int(token[i+3:i+4],16)); i+=5
                        except: byte_list.extend(token[i].encode('utf-8')); i+=1
                    else: byte_list.extend(token[i].encode('utf-8')); i+=1
        try: return bytes(byte_list).decode('utf-8',errors='replace')
        except: return ''

    def save(self, path=None):
        path = path or PATHS['tokenizer_file']
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path,'w',encoding='utf-8') as f:
            json.dump({'vocab_size':len(self.token_to_id),'merges':self.merges,
                       'token_to_id':self.token_to_id,'special_tokens':self.special_tokens},f,ensure_ascii=False,indent=2)
        print(f'Saved tokenizer to {path}')

    def load(self, path=None):
        path = path or PATHS['tokenizer_file']
        with open(path,'r',encoding='utf-8') as f: data=json.load(f)
        self.merges=[tuple(m) for m in data['merges']]
        self.token_to_id=data['token_to_id']
        self.id_to_token={int(v):k for k,v in self.token_to_id.items()}
        if 'special_tokens' in data: self.special_tokens=data['special_tokens']
        self._initialized=True
        print(f'Loaded tokenizer: {len(self.token_to_id)} tokens')

    def __len__(self): return len(self.token_to_id)

def format_input(prompt): return f"{SPECIAL_TOKENS['bos']} {prompt} {SPECIAL_TOKENS['eos']}"
def format_output(output): return f"{output}{SPECIAL_TOKENS['eos']}"
def format_training_pair(sample): return f"{format_input(sample['prompt'])} {format_output(sample['output'])}"

print('Training tokenizer...')
texts = [format_training_pair(s) for s in all_samples]
tokenizer = BPETokenizer(vocab_size=MODEL_CONFIG['vocab_size'])
tokenizer.train(texts)
tokenizer.save()
MODEL_CONFIG['vocab_size'] = len(tokenizer)

test_text = "What's the weather in Beijing?"
enc = tokenizer.encode(test_text)
dec = tokenizer.decode(enc)
print(f'Encode/Decode test: "{test_text}" -> {len(enc)} tokens -> "{dec}"')

## Step 4: Dataset & DataLoader

In [ ]:
class FunctionCallDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        sample = self.samples[idx]
        full_text = format_training_pair(sample)
        token_ids = self.tokenizer.encode(full_text)[:self.max_len]
        prompt_ids = self.tokenizer.encode(format_input(sample['prompt']))
        prompt_len = min(len(prompt_ids), self.max_len)
        pad_len = self.max_len - len(token_ids)
        input_ids = token_ids + [0]*pad_len
        attention_mask = [1]*len(token_ids) + [0]*pad_len
        labels = ([-100]*prompt_len + token_ids[prompt_len:])[:self.max_len]
        labels += [-100]*(self.max_len - len(labels))
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }

train_dataset = FunctionCallDataset(train_samples, tokenizer, MODEL_CONFIG['max_seq_len'])
val_dataset = FunctionCallDataset(val_samples, tokenizer, MODEL_CONFIG['max_seq_len'])
train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                          shuffle=True, num_workers=TRAIN_CONFIG['num_workers'], pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=TRAIN_CONFIG['batch_size'],
                        shuffle=False, num_workers=TRAIN_CONFIG['num_workers'], pin_memory=True)
print(f'DataLoaders: {len(train_loader)} train batches, {len(val_loader)} val batches')

## Step 5: Transformer Model (From Scratch)
GPT-style decoder-only with RoPE, RMSNorm, SwiGLU, KV-Cache.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos().unsqueeze(0))
        self.register_buffer('sin_cached', emb.sin().unsqueeze(0))
    def forward(self, x, seq_len=None):
        return self.cos_cached[:,:seq_len,:], self.sin_cached[:,:seq_len,:]

def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    return (q*cos)+(rotate_half(q)*sin), (k*cos)+(rotate_half(k)*sin)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().pow(2).mean(-1,keepdim=True)+self.eps)).type_as(x) * self.weight

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads, self.head_dim = n_heads, d_model//n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.rotary_emb = RotaryPositionalEmbedding(self.head_dim)
    def forward(self, x, attention_mask=None, past_kv=None, use_cache=False):
        B,S,D = x.shape
        q = self.q_proj(x).view(B,S,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,S,self.n_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,S,self.n_heads,self.head_dim).transpose(1,2)
        cos,sin = self.rotary_emb(x,seq_len=S)
        q,k = apply_rotary_pos_emb(q,k,cos,sin)
        if past_kv is not None:
            k = torch.cat([past_kv[0],k],dim=2)
            v = torch.cat([past_kv[1],v],dim=2)
        new_kv = (k,v) if use_cache else None
        attn_w = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_dim)
        if attention_mask is not None:
            if attention_mask.dim()==2: attention_mask=attention_mask.unsqueeze(1).unsqueeze(2)
            attn_w = attn_w.masked_fill(attention_mask==0, float('-inf'))
        sq,sk = q.shape[2],k.shape[2]
        causal = torch.triu(torch.ones(sq,sk,device=x.device,dtype=torch.bool), diagonal=sk-sq+1)
        attn_w = attn_w.masked_fill(causal.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_w = F.softmax(attn_w, dim=-1, dtype=torch.float32).type_as(q)
        attn_w = self.dropout(attn_w)
        out = torch.matmul(attn_w,v).transpose(1,2).contiguous().view(B,S,D)
        return self.out_proj(out), new_kv

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x): return self.dropout(self.w2(F.silu(self.w1(x))*self.w3(x)))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff_norm = RMSNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
    def forward(self, x, attention_mask=None, past_kv=None, use_cache=False):
        h, kv = self.attn(self.attn_norm(x), attention_mask, past_kv, use_cache)
        x = x + h
        x = x + self.ff(self.ff_norm(x))
        return x, kv

class GPTModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        config = config or MODEL_CONFIG
        self.config = config
        self.token_embedding = nn.Embedding(config['vocab_size'], config['d_model'])
        self.dropout = nn.Dropout(config['dropout'])
        self.layers = nn.ModuleList([TransformerBlock(config['d_model'],config['n_heads'],
                                                       config['d_ff'],config['dropout'])
                                      for _ in range(config['n_layers'])])
        self.final_norm = RMSNorm(config['d_model'])
        self.lm_head = nn.Linear(config['d_model'], config['vocab_size'], bias=False)
        self.token_embedding.weight = self.lm_head.weight
        self.apply(self._init_weights)
        n = sum(p.numel() for p in self.parameters())
        print(f'GPTModel: {n/1e6:.2f}M params')

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, attention_mask=None, labels=None, past_key_values=None, use_cache=False):
        x = self.dropout(self.token_embedding(input_ids))
        new_kvs = [] if use_cache else None
        for i, layer in enumerate(self.layers):
            pk = past_key_values[i] if past_key_values is not None else None
            x, kv = layer(x, attention_mask, pk, use_cache)
            if use_cache: new_kvs.append(kv)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if labels is not None:
            sl, stl = logits[:,:-1,:].contiguous(), labels[:,1:].contiguous()
            loss = F.cross_entropy(sl.view(-1,sl.size(-1)), stl.view(-1), ignore_index=-100)
        return {'logits':logits, 'loss':loss, 'past_key_values':new_kvs}

    @torch.no_grad()
    def generate(self, input_ids, tokenizer, max_new_tokens=128, temperature=0.1,
                 top_p=0.9, repetition_penalty=1.2, attention_mask=None):
        self.eval()
        device = next(self.parameters()).device
        if input_ids.dim()==1: input_ids=input_ids.unsqueeze(0)
        input_ids = input_ids.to(device)
        if attention_mask is None: attention_mask=torch.ones_like(input_ids)
        attention_mask = attention_mask.to(device)
        past_kv, generated = None, input_ids
        for _ in range(max_new_tokens):
            if past_kv is None:
                out = self(generated, attention_mask=attention_mask, use_cache=True)
            else:
                out = self(generated[:,-1:], past_key_values=past_kv, use_cache=True)
            logits, past_kv = out['logits'][:,-1,:], out['past_key_values']
            if repetition_penalty > 1.0:
                for tid in generated[0].tolist():
                    logits[0,tid] = logits[0,tid]/repetition_penalty if logits[0,tid]>0 else logits[0,tid]*repetition_penalty
            if temperature > 0:
                probs = F.softmax(logits/temperature, dim=-1)
                sp,si = torch.sort(probs, descending=True)
                cp = torch.cumsum(sp, dim=-1)
                sp[cp-sp>top_p] = 0
                sp = sp/sp.sum()
                nt = si.gather(-1, torch.multinomial(sp,1))
            else: nt = logits.argmax(dim=-1,keepdim=True)
            generated = torch.cat([generated,nt],dim=-1)
            if nt.item()==tokenizer.eos_token_id: break
        return generated[0].tolist()

model = GPTModel(MODEL_CONFIG)
n = sum(p.numel() for p in model.parameters())
print(f'Model created: {n/1e6:.2f}M params, {MODEL_CONFIG["n_layers"]}L x {MODEL_CONFIG["d_model"]}D')

## Step 6: Training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel: {n_gpus} GPUs')
model.to(device)

optimizer = AdamW(model.parameters(), lr=TRAIN_CONFIG['learning_rate'],
                  weight_decay=TRAIN_CONFIG['weight_decay'], betas=(0.9,0.95))
total_steps = len(train_loader) * TRAIN_CONFIG['epochs']

def cosine_lr(step):
    ws = TRAIN_CONFIG['warmup_steps']
    if step < ws: return float(step)/max(1,ws)
    p = float(step-ws)/max(1,total_steps-ws)
    return max(0.0, 0.5*(1.0+math.cos(math.pi*p)))

scheduler = LambdaLR(optimizer, cosine_lr)
scaler = torch.amp.GradScaler('cuda', enabled=TRAIN_CONFIG['fp16'])
print(f'Training: {TRAIN_CONFIG["epochs"]} epochs, {total_steps} steps, LR={TRAIN_CONFIG["learning_rate"]}')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss, steps = 0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(loader):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbl = batch['labels'].to(device)
        with torch.amp.autocast('cuda', enabled=TRAIN_CONFIG['fp16']):
            loss = model(input_ids=ids, attention_mask=mask, labels=lbl)['loss']
        if loss.dim() > 0:
            loss = loss.mean()
        if TRAIN_CONFIG['fp16']:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['max_grad_norm'])
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG['max_grad_norm'])
            optimizer.step()
        optimizer.zero_grad(); scheduler.step()
        total_loss += loss.item(); steps += 1
        if (step+1)%50==0:
            print(f'  E{epoch} S{step+1}/{len(loader)} | Loss:{total_loss/steps:.4f} | LR:{optimizer.param_groups[0]["lr"]:.2e}')
    return total_loss/steps

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, steps = 0, 0
    for batch in loader:
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbl = batch['labels'].to(device)
        with torch.amp.autocast('cuda', enabled=TRAIN_CONFIG['fp16']):
            total_loss += model(input_ids=ids, attention_mask=mask, labels=lbl)['loss'].mean().item()
        steps += 1
    avg = total_loss/steps
    return avg, math.exp(min(avg,20))

best_val_loss = float('inf')
print('='*60)
print('TRAINING START')
print('='*60)
for epoch in range(1, TRAIN_CONFIG['epochs']+1):
    t0 = time.time()
    tl = train_epoch(model, train_loader, optimizer, scheduler, scaler, device, epoch)
    vl, vp = evaluate(model, val_loader, device)
    elapsed = time.time()-t0
    print(f'\nE{epoch}: train={tl:.4f} val={vl:.4f} ppl={vp:.2f} time={elapsed:.1f}s')
    raw = model.module if hasattr(model,'module') else model
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save({'model_state_dict':raw.state_dict(),'model_config':raw.config,
                    'epoch':epoch,'best_loss':best_val_loss}, PATHS['best_model'])
        print(f'  *** Best model saved! val_loss={best_val_loss:.4f} ***')
    torch.cuda.empty_cache()

print(f'\nTRAINING DONE! Best val_loss={best_val_loss:.4f}')

## Step 7: Inference Test

In [ ]:
raw_model = model.module if hasattr(model,'module') else model
raw_model.eval()

def run_inference(prompt, model=raw_model, tok=tokenizer, dev=device, max_tokens=64, temp=0.1):
    input_text = format_input(prompt)
    input_ids = torch.tensor([tok.encode(input_text)], dtype=torch.long).to(dev)
    output_ids = model.generate(input_ids, tok, max_new_tokens=max_tokens,
                                temperature=temp, top_p=0.9, repetition_penalty=1.2)
    decoded = tok.decode(output_ids)
    if '<eos>' in decoded:
        parts = decoded.split('<eos>')
        result = parts[1].strip() if len(parts)>1 else decoded.replace('<bos>','').replace('<eos>','').strip()
    else:
        result = decoded.replace('<bos>','').strip()
    if '<eos>' in result: result = result.split('<eos>')[0].strip()
    return result

test_prompts = [
    "What's the weather in Beijing?",
    "List files in /home/user.",
    "How is the weather in Tokyo?",
    "Show me the contents of /var/log.",
    "Can you tell me the weather in London?",
    "What files are in /tmp?",
    "Is it raining in Paris?",
    "Browse /data/projects.",
]

print('='*60)
print('INFERENCE TEST')
print('='*60)
for p in test_prompts:
    out = run_inference(p)
    print(f'\n  Q: {p}')
    print(f'  A: {out}')

## Step 8: Tool Execution & Full Pipeline

In [ ]:
def execute_tool_call(tool_call_str):
    try:
        tc = json.loads(tool_call_str)
        fn, args = tc.get('name',''), tc.get('arguments',{})
        if fn == 'get_weather':
            return json.dumps({'city':args.get('city','?'),'temperature':'22C',
                'condition':'Partly Cloudy','humidity':'65%','wind':'12 km/h'}, indent=2)
        elif fn == 'list_directory':
            return json.dumps({'path':args.get('path','/'),'contents':[
                {'name':'documents','type':'directory'},{'name':'downloads','type':'directory'},
                {'name':'readme.txt','type':'file','size':'1.2KB'},
                {'name':'config.json','type':'file','size':'0.5KB'},
                {'name':'data.csv','type':'file','size':'3.8KB'}]}, indent=2)
        return json.dumps({'error':f'Unknown: {fn}'})
    except json.JSONDecodeError:
        return json.dumps({'error':'Invalid format'})

def full_pipeline(prompt):
    raw = run_inference(prompt)
    try:
        tc = json.loads(raw)
        fn, args = tc.get('name',''), tc.get('arguments',{})
        result = execute_tool_call(raw)
        return {'function_call':tc, 'tool_result':json.loads(result), 'raw':raw}
    except (json.JSONDecodeError, KeyError):
        return {'function_call':None, 'tool_result':None, 'raw':raw}

for p in ["What's the weather in Shanghai?", "List files in /home/user/documents."]:
    r = full_pipeline(p)
    print(f'\nQ: {p}')
    if r['function_call']:
        print(f'  Function: {r["function_call"]["name"]}')
        print(f'  Args: {r["function_call"]["arguments"]}')
        print(f'  Result: {json.dumps(r["tool_result"], indent=4)}')
    else:
        print(f'  Raw: {r["raw"]}')

## Step 9: OpenAI-Compatible API Server

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
import uvicorn, threading

app = FastAPI(title='Custom LLM API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True,
                   allow_methods=['*'], allow_headers=['*'])

class ChatMessage(BaseModel):
    role: str; content: str
class ChatReq(BaseModel):
    model: str = 'custom-llm'
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.1
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 128
    stream: Optional[bool] = False

@app.get('/v1/models')
async def list_models():
    return {'object':'list','data':[{'id':'custom-llm','object':'model','created':int(time.time()),'owned_by':'custom'}]}

@app.post('/v1/chat/completions')
async def chat_completions(req: ChatReq):
    um = next((m.content for m in req.messages if m.role=='user'), '')
    if not um: raise HTTPException(400, 'No user message')
    try:
        r = full_pipeline(um)
        if r['function_call']:
            fn = r['function_call']['name']
            args = json.dumps(r['function_call']['arguments'])
            tr = json.dumps(r['tool_result'], indent=2)
            content = f'Function called: {fn}\nArguments: {args}\n\nResult:\n{tr}'
        else:
            content = r['raw'] or 'Could not understand.'
        return {'id':f'chatcmpl-{uuid.uuid4().hex[:12]}','object':'chat.completion',
                'created':int(time.time()),'model':req.model,
                'choices':[{'index':0,'message':{'role':'assistant','content':content},'finish_reason':'stop'}],
                'usage':{'prompt_tokens':len(tokenizer.encode(um)),
                         'completion_tokens':len(tokenizer.encode(content)),
                         'total_tokens':len(tokenizer.encode(um))+len(tokenizer.encode(content))}}
    except Exception as e:
        raise HTTPException(500, str(e))

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print('Server running at http://0.0.0.0:8000')
print('Endpoints: GET /v1/models, POST /v1/chat/completions')

## Step 10: Test API

In [ ]:
import urllib.request, time
time.sleep(2)

def test_api(prompt):
    data = json.dumps({'model':'custom-llm','messages':[{'role':'user','content':prompt}],
                       'temperature':0.1,'max_tokens':128}).encode()
    req = urllib.request.Request('http://localhost:8000/v1/chat/completions',
                                data=data, headers={'Content-Type':'application/json'})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

for p in ["What's the weather in Beijing?", "List files in /home/user."]:
    r = test_api(p)
    print(f'\nQ: {p}')
    print(f'A: {r["choices"][0]["message"]["content"]}')
    print(f'Tokens: {r["usage"]}')

## Step 11: Save & Export

In [ ]:
print('All artifacts saved:')
print(f'  Model: {PATHS["best_model"]}')
print(f'  Tokenizer: {PATHS["tokenizer_file"]}')
print(f'  Dataset: {PATHS["data_dir"]}')
print(f'\nFile sizes:')
for k,v in PATHS.items():
    if os.path.exists(v) and os.path.isfile(v):
        sz = os.path.getsize(v)
        print(f'  {k}: {sz/1e6:.1f}MB' if sz>1e6 else f'  {k}: {sz/1e3:.1f}KB')

print('\n' + '='*60)
print('PROJECT COMPLETE!')
print('='*60)
print('\nTo use the API server externally:')
print('  1. Run: python server.py')
print('  2. Open: http://localhost:8000')
print('  3. API: POST http://localhost:8000/v1/chat/completions')
print('\nOpenAI-compatible curl example:')
print('  curl -X POST http://localhost:8000/v1/chat/completions \\')
print('    -H "Content-Type: application/json" \\')
print('    -d \'{"model":"custom-llm","messages":[{"role":"user","content":"What\'s the weather in Beijing?"}]}\'')